### 接入自定义工具函数
LangGraph接入工具函数类别和LangChain一样有三类：LangChain内置的工具函数、自定义工具函数和使用MCP工具。我们这里通过自定义工具函数进行演示。

1. 创建自定义获取天气信息的工具函数，为了更加严谨，我们使用pydantic库定义一个对象类型描述传入参数，这里表示要传入的是一个字符串 city 参数，表示的含义是城市名称。定义的WeatherQuery对象在@tool(args_schema=WeatherQuery)中约束get_weather的函数参数。tool装饰器可以将自定义函数修饰为LangChain/LangGraph的函数工具， 注意函数的注释必须要撰写清楚才能使大模型理解函数功能。
tools.py 文件内容如下：

In [8]:
import json
import os
import httpx
import dotenv
from loguru import logger
from pydantic import Field, BaseModel
from langchain_core.tools import tool

# 加载环境变量配置
dotenv.load_dotenv()

class WeatherQuery(BaseModel):
    """
    天气查询参数模型类，用于定义天气查询工具的输入参数结构。
    
    :param city: 城市名称，字符串类型，表示要查询天气的城市
    """
    city: str = Field(description="城市名称")

@tool(args_schema=WeatherQuery)
def get_weather(city):
    """
    查询指定城市的即时天气信息。

    :param city: 必要参数，字符串类型，表示要查询天气的城市名称。
                 注意：中国城市需使用其英文名称，如 "Beijing" 表示北京。
    :return: 返回 OpenWeather API 的响应结果，URL 为
             https://api.openweathermap.org/data/2.5/weather。
             响应内容为 JSON 格式的字符串，包含详细的天气数据。
    """
    # 构建请求 URL
    url = "https://api.openweathermap.org/data/2.5/weather"

    # 设置查询参数
    params = {
        "q": city, # 城市名称
        "appid": os.getenv("OPENWEATHER_API_KEY"),  # 从环境变量中读取 API Key
        "units": "metric",  # 使用摄氏度作为温度单位
        "lang": "zh_cn"     # 返回简体中文的天气描述
    }

    # 发送 GET 请求并获取响应
    response = httpx.get(url, params=params)

    # 将响应解析为 JSON 并序列化为字符串返回
    data = response.json()
    logger.info(f"查询天气结果：{json.dumps(data)}")
    return json.dumps(data)

print(get_weather.name)
print(get_weather.description)
print(get_weather.args)

get_weather
查询指定城市的即时天气信息。

:param city: 必要参数，字符串类型，表示要查询天气的城市名称。
             注意：中国城市需使用其英文名称，如 "Beijing" 表示北京。
:return: 返回 OpenWeather API 的响应结果，URL 为
         https://api.openweathermap.org/data/2.5/weather。
         响应内容为 JSON 格式的字符串，包含详细的天气数据。
{'city': {'description': '城市名称', 'title': 'City', 'type': 'string'}}


2. 创建 LangGraph 智能体。初始化大模型和函数列表并创建ReACT预制图结构并构建智能体。

In [11]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
import os
import dotenv
from langchain.chat_models import init_chat_model
dotenv.load_dotenv(override=True)

# 初始化大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# 定义工具列表，包含天气查询工具
tools = [get_weather]

# 创建ReAct代理，结合语言模型和工具函数
agent = create_react_agent(model=llm, tools=tools)

# 调用代理处理用户查询，获取北京天气信息
response = agent.invoke({"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]})
# 输出完整响应结果和最终回答内容
print(response)
print(response["messages"][-1].content)
response["messages"][-1].pretty_print()
# 使用stream方法进行流式调用
for chunk in agent.stream(
        {"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]},
        stream_mode="values",
):
    chunk["messages"][-1].pretty_print()
# 这里stream_mode有四种选项：
# - messages：流式输出大语言模型回复的token
# - updates : 流式输出每个工具调用的每个步骤。
# - values : 一次输出到所有的chunk。默认值。
# - custom : 自定义输出。主要是可以在工具内部使用get_stream_writer获取输入流，添加自定义的内容。

C:\Users\zhanghailong\AppData\Local\Temp\ipykernel_12172\1030659858.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools)
2026-05-18 10:28:42.329 | INFO     | __main__:get_weather:47 - 查询天气结果：{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "\u9634\uff0c\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 23.69, "feels_like": 23.47, "temp_min": 23.69, "temp_max": 23.69, "pressure": 1010, "humidity": 52, "sea_level": 1010, "grnd_level": 1005}, "visibility": 10000, "wind": {"speed": 2.71, "deg": 23, "gust": 4.63}, "clouds": {"all": 100}, "dt": 1779070694, "sys": {"country": "CN", "sunrise": 1779051441, "sunset": 1779103472}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}


{'messages': [HumanMessage(content='请问北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='2cc6bf8b-5adb-4514-bdac-76dad1ed508c'), AIMessage(content='好的，我来查询北京今天的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 372, 'total_tokens': 425, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 116}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b1de41b9-5da8-43d9-939f-f067ed4ba1b4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e38ea-0084-7971-ad51-a34f051abbb7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Beijing'}, 'id': 'call_00_nauE0LWunTpki0qaRnuR8321', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 372, 'output_tokens': 53, 'total_tokens': 4

2026-05-18 10:28:47.209 | INFO     | __main__:get_weather:47 - 查询天气结果：{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "\u9634\uff0c\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 23.69, "feels_like": 23.47, "temp_min": 23.69, "temp_max": 23.69, "pressure": 1010, "humidity": 52, "sea_level": 1010, "grnd_level": 1005}, "visibility": 10000, "wind": {"speed": 2.71, "deg": 23, "gust": 4.63}, "clouds": {"all": 100}, "dt": 1779070694, "sys": {"country": "CN", "sunrise": 1779051441, "sunset": 1779103472}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}


================================= Tool Message =================================
Name: get_weather

{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "\u9634\uff0c\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 23.69, "feels_like": 23.47, "temp_min": 23.69, "temp_max": 23.69, "pressure": 1010, "humidity": 52, "sea_level": 1010, "grnd_level": 1005}, "visibility": 10000, "wind": {"speed": 2.71, "deg": 23, "gust": 4.63}, "clouds": {"all": 100}, "dt": 1779070694, "sys": {"country": "CN", "sunrise": 1779051441, "sunset": 1779103472}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}
================================== Ai Message ==================================

北京今天的天气情况如下：

🌤 **天气状况**：阴，多云
🌡 **当前温度**：23.69°C（体感温度 23.47°C）
💧 **湿度**：52%
🌬 **风速**：2.71 m/s（约 3 级风，东北风）
👁 **能见度**：10公里（良好）
☁ **云量**：100%（阴天）

整体来看，今天北京天气以**阴天多云**为主，温度适中，大约在 **23~24°C** 左右，体感比较舒适，风力不大。建议出门可适当添件薄外套，以防阴天体感偏凉。


## ReAct Agent 外部工具调用形式
### 添加多个工具函数
添加get_weather和write_file两个工具函数，分别用来查询天气和保存内容至文件。完整的项目代码如下:

In [12]:
import json
import os
import httpx
import dotenv
from loguru import logger
from pydantic import Field, BaseModel
from langchain_core.tools import tool

# 加载环境变量配置
dotenv.load_dotenv()

#规范工具参数
class WeatherQuery(BaseModel):
    """
    天气查询参数模型类，用于定义天气查询工具的输入参数结构。

    :param city: 城市名称，字符串类型，表示要查询天气的城市
    """
    city: str = Field(description="城市名称")


class WriteQuery(BaseModel):
    """
    写入查询模型类
    
    用于定义需要写入文档的内容结构，继承自BaseModel基类
    
    属性:
        content (str): 需要写入文档的具体内容，包含详细的描述信息
    """
    content: str = Field(description="需要写入文档的具体内容")



@tool(args_schema=WeatherQuery)
def get_weather(city):
    """
    查询指定城市的即时天气信息。

    :param city: 必要参数，字符串类型，表示要查询天气的城市名称。
                 注意：中国城市需使用其英文名称，如 "Beijing" 表示北京。
    :return: 返回 OpenWeather API 的响应结果，URL 为
             https://api.openweathermap.org/data/2.5/weather。
             响应内容为 JSON 格式的字符串，包含详细的天气数据。
    """
    # 构建请求 URL
    url = "https://api.openweathermap.org/data/2.5/weather"

    # 设置查询参数
    params = {
        "q": city,  # 城市名称
        "appid": os.getenv("OPENWEATHER_API_KEY"),  # 从环境变量中读取 API Key
        "units": "metric",  # 使用摄氏度作为温度单位
        "lang": "zh_cn"  # 返回简体中文的天气描述
    }

    # 发送 GET 请求并获取响应
    response = httpx.get(url, params=params)

    # 将响应解析为 JSON 并序列化为字符串返回
    data = response.json()
    logger.info(f"查询天气结果：{json.dumps(data)}")
    return json.dumps(data)


@tool(args_schema=WriteQuery)
def write_file(content):
    """
    将指定内容写入本地文件
    
    参数:
        content (str): 要写入文件的文本内容
    
    返回值:
        str: 表示写入操作成功完成的提示信息
    """
    # 将内容写入res.txt文件，使用utf-8编码确保中文字符正确保存
    with open('res.txt', 'w', encoding='utf-8') as f:
        f.write(content)
        logger.info(f"已成功写入本地文件，写入内容：{content}")
        return "已成功写入本地文件。"

### 工具并联调用
测试ReAct图智能体的工具并联调用，同时查询北京和杭州的天气。

In [16]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
import os
import dotenv
from langchain.chat_models import init_chat_model
dotenv.load_dotenv(override=True)

# 初始化大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# 定义工具列表，包含天气查询和结果写入工具
tools = [get_weather, write_file]

# 创建ReAct代理，结合语言模型和工具函数
agent = create_react_agent(model=llm, tools=tools)

# 调用代理处理用户查询，获取北京天气信息
response = agent.invoke({"messages": [{"role": "user", "content": "请问北京和上海今天谁更热？"}]})

# 输出完整响应结果和最终回答内容
print(response)
response["messages"][-1].pretty_print()

C:\Users\zhanghailong\AppData\Local\Temp\ipykernel_12172\2737708076.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools)
2026-05-18 10:38:37.869 | INFO     | __main__:get_weather:62 - 查询天气结果：{"coord": {"lon": 121.4581, "lat": 31.2222}, "weather": [{"id": 803, "main": "Clouds", "description": "\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 27.05, "feels_like": 27.46, "temp_min": 27.05, "temp_max": 27.05, "pressure": 1013, "humidity": 50, "sea_level": 1013, "grnd_level": 1012}, "visibility": 10000, "wind": {"speed": 7.22, "deg": 160, "gust": 10.57}, "clouds": {"all": 81}, "dt": 1779071160, "sys": {"country": "CN", "sunrise": 1779051420, "sunset": 1779101063}, "timezone": 28800, "id": 1796236, "name": "Shanghai", "cod": 200}
2026-05-18 10:38:38.180 | I

{'messages': [HumanMessage(content='请问北京和上海今天谁更热？', additional_kwargs={}, response_metadata={}, id='e581a149-cc45-47a3-8da4-44ffbc8316c6'), AIMessage(content='好的，我来查询北京和上海今天的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 460, 'total_tokens': 547, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 76}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '8f697ef9-8d46-40b7-8f60-f59c4a59af00', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e38f3-156b-75f0-ae06-0b63d2b3b376-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Beijing'}, 'id': 'call_00_RvlPNYRPjKrYl1SqePwz3787', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Shanghai'}, 'id': 'call_01_vIJThKoR5SXbdGbZ9AAG7188',

### 工具串联调用
同时查询北京和上海天气，并将结果保存到文件中

In [17]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# 定义工具列表，包含天气查询和结果写入工具
tools = [get_weather, write_file]

# 创建ReAct代理，结合语言模型和工具函数
agent = create_react_agent(model=llm, tools=tools)

# 调用代理处理用户查询，获取北京天气信息
response = agent.invoke({"messages": [{"role": "user", "content": "请问北京天气怎么样？然后把回答结果写入文件。"}]})
# 输出完整响应结果和最终回答内容
print(response)
response["messages"][-1].pretty_print()

C:\Users\zhanghailong\AppData\Local\Temp\ipykernel_12172\2185275189.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools)
2026-05-18 10:40:23.498 | INFO     | __main__:get_weather:62 - 查询天气结果：{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 803, "main": "Clouds", "description": "\u591a\u4e91", "icon": "04d"}], "base": "stations", "main": {"temp": 25.87, "feels_like": 25.56, "temp_min": 25.87, "temp_max": 25.87, "pressure": 1010, "humidity": 40, "sea_level": 1010, "grnd_level": 1005}, "visibility": 10000, "wind": {"speed": 2.88, "deg": 29, "gust": 4.06}, "clouds": {"all": 77}, "dt": 1779071566, "sys": {"country": "CN", "sunrise": 1779051441, "sunset": 1779103472}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}
2026-05-18 10:40:26.407 | INFO 

{'messages': [HumanMessage(content='请问北京天气怎么样？然后把回答结果写入文件。', additional_kwargs={}, response_metadata={}, id='a0511d30-e74d-4f2a-b9ea-f3424c1774cf'), AIMessage(content='好的，我来查一下北京的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 462, 'total_tokens': 515, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 78}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '8c31d499-554b-4b6c-8dfb-e354290d7709', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e38f4-b09b-74e3-856b-0d94ecd7a96a-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Beijing'}, 'id': 'call_00_2OQgR6hT9g7Nqfip9L1G0841', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 462, 'output_tokens': 53, 'total_t